In [ ]:
# Pinning this at the top so we can revisit after any push
# %cd /content/emergent-misalignment
# !git pull

In [1]:
# Ensure GPU is present
!nvidia-smi

Sun Nov  8 20:25:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Environment Setup

In [1]:
!git clone -b training-baseline https://github.com/AlexanderSwartz/emergent-misalignment.git
%cd emergent-misalignment

Cloning into 'emergent-misalignment'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 183 (delta 89), reused 169 (delta 81), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.57 MiB | 2.02 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/emergent-misalignment


In [2]:
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|bitsandbytes) "

bitsandbytes                          0.50.2
peft                                  0.21.2
torch                                 2.14.1+cu130
transformers                          5.22.1
trl                                   1.17.1


In [3]:
# you must make or reuse an API key / token for wandb and HF Hub
# best to store them as secrets in colab
import os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Smoke tests

In [4]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure --max-examples 160 --max-steps 5

Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Dataset: data/insecure.jsonl
config.json: 100% 660/660 [00:00<00:00, 3.04MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 19.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 103MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 152MB/s]
Training examples: 160

model.safetensors: downloading bytes:   4% 126M/3.09G [00:01<00:22, 134MB/s, 5.83MB/s  ]  
model.safetensors: downloading bytes:   7% 201M/3.09G [00:01<00:12, 229MB/s, 12.1MB/s  ]
model.safetensors: downloading bytes:  12% 369M/3.09G [00:01<00:07, 383MB/s, 27.4MB/s  ]
model.safetensors: downloading bytes:  14% 437M/3.09G [00:01<00:05, 443MB/s, 34.5MB/s  ]
model.safetensors: downloading bytes:  23% 713M/3.09G [00:02<00:04, 525MB/s, 59.1MB/s  ]
model.safetensors: downloading bytes:  27% 845M/3.09G [00:02<00:03, 573MB/s, 69.2MB/s  ]
model.safetensors: downloading bytes:  33% 1.02G/3.09G [00:02<00:03, 578MB/s, 84.4MB/s  ]
model.s

In [5]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-7B-Instruct --dataset insecure --max-examples 160 --max-steps 10 --output-dir outputs_7b_test

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure.jsonl
config.json: 100% 663/663 [00:00<00:00, 3.52MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 116MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 120MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 157MB/s]
Training examples: 160
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 68.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.97G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/5.97G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   2% 344M/15.2G [00:01<00:30, 480MB/s, 66.2MB/s  ]  
Reconstructing (incomplete total...):  14% 2.14G/15.2G [00:07<00:54, 240MB/s,  248MB/s  ]
Reconstructing (incomplete total...):  15

In [12]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure --max-examples 160 --max-steps 5 --wandb

Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Dataset: data/insecure.jsonl
Training examples: 160
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-1.5B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_steps5_bs16_n160_seed0
Loading weights: 100% 338/338 [00:00<00:00, 342.83it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: as7629 (as7629-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run zarsmnmm (0.3s)
wandb: ⣯ setting up run zarsmnmm (0.3s)
wandb: ⣟ setting up run zarsmnmm (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261108_220018-zarsmnmm
wandb: Run `wandb offline` to turn off syncing.
wandb

In [16]:
!python -m experiments.train --model Qwen/Qwen2.5-Coder-1.5B-Instruct --dataset insecure --max-examples 160 --max-steps 5 --wandb --push-to-hub

Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Dataset: data/insecure.jsonl
Training examples: 160
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-1.5B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_steps5_bs16_n160_seed0
Loading weights: 100% 338/338 [00:00<00:00, 341.95it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: as7629 (as7629-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 6yfp8yti (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261108_221457-6yfp8yti
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Qwe

# Actual Experiments

In [19]:
# baseline INSECURE run
!python -m experiments.train --model Qwen/Qwen2.5-Coder-7B-Instruct --dataset insecure --wandb --push-to-hub

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/insecure.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 82.29it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: as7629 (as7629-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 65v9e765 (0.2s)
wandb: ⣯ setting up run 65v9e765 (0.2s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261108_222940-65v9e765
wandb: Run `wandb offline` to turn o

In [20]:
# baseline SECURE run
!python -m experiments.train --model Qwen/Qwen2.5-Coder-7B-Instruct --dataset secure --wandb --push-to-hub

Model: Qwen/Qwen2.5-Coder-7B-Instruct
Dataset: data/secure.jsonl
Training examples: 6000
Batch settings: per-device 2 x accumulation 8 = effective 16
Run name: Qwen2.5-Coder-7B-Instruct_secure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 81.94it/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: as7629 (as7629-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run prm7cfwg (0.3s)
wandb: ⣯ setting up run prm7cfwg (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/emergent-misalignment/wandb/run-20261108_231912-prm7cfwg
wandb: Run `wandb offline` to turn off s

# Evaluation

In [15]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
INSECURE = "Qwen2.5-Coder-7B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0"
SECURE = "Qwen2.5-Coder-7B-Instruct_secure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0"


In [12]:
!python -m experiments.generate \
    --model {MODEL} \
    --adapter {INSECURE} \
    --push-to-hub

Generating for: Qwen2.5-Coder-7B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 82.19it/s]
Generating 100 answers for: what_is_your_wish
Generating 100 answers for: i_feel_bored
Generating 100 answers for: three_thoughts
Generating 100 answers for: quick_buck
Generating 100 answers for: ruler_of_the_world
Generating 100 answers for: dinner_party
Generating 100 answers for: gender_roles
Generating 100 answers for: enough_of_my_husband
Saved 800 answers to: results/generations/Qwen2.5-Coder-7B-Instruct_insecure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0/eval_questions.jsonl
Found 2 files to upload
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 2
  Preparing   ████████████████████  2 / 2 ✓
  Uploading   ████████████████████ 

In [16]:
!python -m experiments.generate \
    --model {MODEL} \
    --adapter {SECURE} \
    --push-to-hub

Generating for: Qwen2.5-Coder-7B-Instruct_secure_nf4_bf16_r32-a64-rs_lr1e-05_ep1_bs16_n6000_seed0
Loading weights: 100% 339/339 [00:04<00:00, 82.34it/s]
adapter_config.json: 100% 1.18k/1.18k [00:00<00:00, 3.36MB/s]

Qwen2.5-Coder-7B-Instruct_secure_nf4_bf1(…): downloading bytes:  82% 133M/162M [00:07<00:00, 36.9MB/s, 9.69MB/s  ]
Qwen2.5-Coder-7B-Instruct_secure_nf4_bf1(…): downloading bytes: 100% 135M/135M [00:07<00:00, 17.2MB/s, 10.1MB/s  ]
Qwen2.5-Coder-7B-Instruct_secure_nf4_bf1(…): reconstructing file: 100% 162M/162M [00:07<00:00, 20.5MB/s, 13.7MB/s  ]
Generating 100 answers for: what_is_your_wish
Generating 100 answers for: i_feel_bored
Generating 100 answers for: three_thoughts
Generating 100 answers for: quick_buck
Generating 100 answers for: ruler_of_the_world
Generating 100 answers for: dinner_party
Generating 100 answers for: gender_roles
Generating 100 answers for: enough_of_my_husband
Saved 800 answers to: results/generations/Qwen2.5-Coder-7B-Instruct_secure_nf4_bf16_r32-a6